# Task 4: POS Tagging and Syntactic Parsing

Feature included

- Text preprocessing
- POS tagging
- Action verb extraction
- Keyword extraction
- Named Entity Recognition (NER)
- Syntactic parsing
- Event extraction
- Relationship extraction between entities and events

In [1]:
!pip install nltk pandas spacy
!python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 9.1 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
# Import Libraries

import nltk
import pandas as pd
import spacy
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk import pos_tag, ne_chunk
from nltk.chunk import RegexpParser
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')
nltk.download('stopwords')


nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rahulchauhan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/rahulchauhan/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/rahulchauhan/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /Users/rahulchauhan/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /Users/rahulchauhan/nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     /Users/rahulchauhan/nltk_data...
[nltk_data]   Packa

In [3]:
article_text = """
Apple CEO Tim Cook announced a new AI initiative during an event in California.
The company plans to invest $5 billion over the next three years.
"""

In [4]:
def pos_tagging_component_nltk(article_text):
    """
    Component 2: POS Tagging

    Returns:
        POS tags
        Keywords
        Action verbs
        Named entities
    """

    # -------------------------
    # Tokenization
    # -------------------------

    tokens = word_tokenize(article_text)

    # -------------------------
    # Stopword Removal
    # -------------------------

    stop_words = set(stopwords.words("english"))

    filtered_tokens = [
        word
        for word in tokens
        if word.lower() not in stop_words
        and word.isalnum()
    ]

    # -------------------------
    # POS Tagging
    # -------------------------

    pos_tags = pos_tag(tokens)

    # -------------------------
    # Action Verbs
    # -------------------------

    action_verbs = [
        word
        for word, tag in pos_tags
        if tag.startswith("VB")
    ]

    # -------------------------
    # Keywords
    # -------------------------

    filtered_pos = pos_tag(filtered_tokens)

    keywords = list(set([
        word.lower()
        for word, tag in filtered_pos
        if tag.startswith("NN")
    ]))

    # -------------------------
    # Named Entity Recognition
    # -------------------------

    ner_tree = ne_chunk(pos_tags)

    entities = []

    for subtree in ner_tree:

        if hasattr(subtree, "label"):

            entities.append({
                "Entity": " ".join(
                    word
                    for word, _
                    in subtree.leaves()
                ),
                "Type": subtree.label()
            })

    return {
        "tokens": tokens,
        "pos_tags": pos_tags,
        "keywords": keywords,
        "action_verbs": action_verbs,
        "entities": entities
    }

In [5]:
def syntactic_parsing_component_nltk(article_text):
    """
    Component 3: Syntax & Parsing Engine

    Returns:
        Parse tree
        Noun phrases
        Events
        Relationships
    """

    # -------------------------
    # POS Tagging
    # -------------------------

    tokens = word_tokenize(article_text)

    pos_tags = pos_tag(tokens)

    # -------------------------
    # Grammar
    # -------------------------

    grammar = r"""
    NP: {<DT>?<JJ>*<NN.*>+}
    PP: {<IN><NP>}
    VP: {<VB.*><NP|PP>*}
    """

    parser = RegexpParser(grammar)

    parse_tree = parser.parse(pos_tags)

    # -------------------------
    # Noun Phrases
    # -------------------------

    noun_phrases = []

    for subtree in parse_tree.subtrees():

        if subtree.label() == "NP":

            noun_phrases.append(
                " ".join(
                    word
                    for word, tag
                    in subtree.leaves()
                )
            )

    # -------------------------
    # Events
    # -------------------------

    events = [
        word
        for word, tag in pos_tags
        if tag.startswith("VB")
    ]

    # -------------------------
    # Relationships
    # -------------------------

    relationships = []

    for sentence in sent_tokenize(article_text):

        tagged = pos_tag(word_tokenize(sentence))

        subject = None
        verb = None

        for word, tag in tagged:

            if subject is None and tag.startswith("NN"):
                subject = word

            elif verb is None and tag.startswith("VB"):
                verb = word

        if subject and verb:

            relationships.append({
                "Source_Entity": subject,
                "Related_Event": verb,
                "Object": None
            })
            
    return {
        "noun_phrases": noun_phrases,
        "events": events,
        "relationships": relationships,
        "dependency_structure": parse_tree,
    }

In [6]:
pos_results = pos_tagging_component_nltk(article_text)
parse_results = syntactic_parsing_component_nltk(article_text)

print("========== POS TAGGING RESULTS USING NLTK ==========")

print("\nPos Tags:")
display(
    pd.DataFrame(
        pos_results["pos_tags"],
        columns=["Word","POS_Tag"]
    )
)

print("\n\nEntities:")
display(pd.DataFrame(pos_results["entities"]))

print("\n\nKeywords:", pos_results["keywords"])
print("\n\nAction Verbs:", pos_results["action_verbs"])

print("\n========== SYNTACTIC PARSING RESULTS ==========")

print("\nNoun Phrases:")
print(parse_results["noun_phrases"])

print("\n\nEvents:")
print(parse_results["events"])

print("\n\nRelationships:")
display(pd.DataFrame(parse_results["relationships"]))

========== POS TAGGING RESULTS USING NLTK ==========

Pos Tags:


,Word,POS_Tag
0,Apple,NNP
1,CEO,NNP
2,Tim,NNP
3,Cook,NNP
4,announced,VBD
5,a,DT
6,new,JJ
7,AI,NNP
8,initiative,NN
9,during,IN




Entities:


,Entity,Type
0,Apple,PERSON
1,CEO Tim Cook,ORGANIZATION
2,California,GPE




Keywords: ['tim', 'apple', 'event', 'cook', 'years', 'california', 'ai', 'ceo', 'company']


Action Verbs: ['announced', 'plans', 'invest']

========== SYNTACTIC PARSING RESULTS ==========

Noun Phrases:
['Apple CEO Tim Cook', 'a new AI initiative', 'an event', 'California', 'The company', 'years']


Events:
['announced', 'plans', 'invest']


Relationships:


,Source_Entity,Related_Event,Object
0,Apple,announced,None
1,company,plans,None


In [7]:
def pos_tagging_component_spacy(article_text):
    """
    POS Tagging Component

    Returns:
        POS Tags
        Keywords
        Named Entities
        Action Verbs
    """

    doc = nlp(article_text)

    # ---------------------
    # Tokenization
    # ---------------------
    
    tokens = [
        token.text
        for token in doc
        if not token.is_space
    ]
    
    # ---------------------
    # Stopword Removal
    # ---------------------
    
    filtered_tokens = [
        token.text
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
    ]
    

    # ---------------------
    # POS Tagging
    # ---------------------

    pos_tags = []

    for token in doc:
        if token.is_space:
            continue

        pos_tags.append({
            "Word": token.text,
            "POS_Tag": token.pos_,
            "Detailed_Tag": token.tag_
        })

    # ---------------------
    # Keywords
    # ---------------------

    keywords = list(set([
        token.lemma_.lower()
        for token in doc
        if token.pos_ in ["NOUN", "PROPN"]
        and not token.is_stop
    ]))

    # ---------------------
    # Action Verbs
    # ---------------------

    action_verbs = list(set([
        token.lemma_
        for token in doc
        if token.pos_ == "VERB"
    ]))

    # ---------------------
    # Named Entity Recognition
    # ---------------------

    entities = []

    for ent in doc.ents:

        entities.append({
            "Entity": ent.text,
            "Type": ent.label_
        })

    return {
        "pos_tags": pos_tags,
        "keywords": keywords,
        "action_verbs": action_verbs,
        "entities": entities
    }

In [8]:
def syntactic_parsing_component_spacy(article_text):
    """
    Syntactic Parsing Component

    Returns:
        Noun Phrases
        Events
        Relationships
        Dependency Structure
    """

    doc = nlp(article_text)

    # ---------------------
    # Noun Phrases
    # ---------------------

    noun_phrases = [
        chunk.text.strip()
        for chunk in doc.noun_chunks
    ]

    # ---------------------
    # Events
    # ---------------------

    events = [
        token.lemma_
        for token in doc
        if token.pos_ == "VERB"
    ]

    # ---------------------
    # Relationships
    # ---------------------
    relationships = []
    
    for token in doc:
        if token.is_space:
            continue
    
        if token.pos_ == "VERB":
    
            subject = None
            obj = None
    
            for child in token.children:
    
                if child.dep_ in ["nsubj", "nsubjpass"]:

                    subject = " ".join(
                        tok.text
                        for tok in child.subtree
                        if not tok.is_space
                    )
                
                    subject = " ".join(subject.split())
                
                if child.dep_ in ["dobj", "obj", "attr"]:
                
                    obj = " ".join(
                        tok.text
                        for tok in child.subtree
                        if not tok.is_space
                    )
                
                    obj = " ".join(obj.split())
    
            if subject:
    
                relationships.append({
                    "Source_Entity": subject,
                    "Related_Event": token.lemma_,
                    "Object": obj
                })
    

    # ---------------------
    # Dependency Structure
    # ---------------------

    dependencies = []

    for token in doc:

        dependencies.append({
            "Word": token.text,
            "Dependency": token.dep_,
            "Head": token.head.text
        })

    return {
        "noun_phrases": noun_phrases,
        "events": events,
        "relationships": relationships,
        "dependency_structure": dependencies
    }

In [9]:
pos_results = pos_tagging_component_spacy(article_text)
parse_results = syntactic_parsing_component_spacy(article_text)

print("========== POS TAGGING ==========")

display(pd.DataFrame(pos_results["pos_tags"]))

print("\nEntities")
display(pd.DataFrame(pos_results["entities"]))

print("\nKeywords")
print(pos_results["keywords"])

print("\nAction Verbs")
print(pos_results["action_verbs"])

print("\n========== PARSING ==========")

print("\nNoun Phrases")
print(parse_results["noun_phrases"])

print("\nEvents")
print(parse_results["events"])

print("\nRelationships")
display(pd.DataFrame(parse_results["relationships"]))

========== POS TAGGING ==========


,Word,POS_Tag,Detailed_Tag
0,Apple,PROPN,NNP
1,CEO,PROPN,NNP
2,Tim,PROPN,NNP
3,Cook,PROPN,NNP
4,announced,VERB,VBD
5,a,DET,DT
6,new,ADJ,JJ
7,AI,PROPN,NNP
8,initiative,NOUN,NN
9,during,ADP,IN



Entities


,Entity,Type
0,Apple,ORG
1,Tim Cook,PERSON
2,AI,GPE
3,California,GPE
4,$5 billion,MONEY
5,the next three years,DATE



Keywords
['tim', 'apple', 'event', 'cook', 'initiative', 'california', 'year', 'ai', 'ceo', 'company']

Action Verbs
['announce', 'plan', 'invest']

========== PARSING ==========

Noun Phrases
['Apple CEO Tim Cook', 'a new AI initiative', 'an event', 'California', 'The company', 'the next three years']

Events
['announce', 'plan', 'invest']

Relationships


,Source_Entity,Related_Event,Object
0,Apple CEO Tim Cook,announce,a new AI initiative
1,The company,plan,None
